# Next-hour cloud map using satellite maps and weather stations

Run these cells in order in VS Code or Jupyter. In the Run cell, set `DATA_DIR` to the folder containing your data files.

Place the 2024 cloud `.npz` file and these five station CSV files anywhere
beneath `--data-dir`:
`hly3723_subset.csv`, `hly3904_subset.csv`, `hly4935_subset.csv`,
`hly518_subset.csv`, and `hly532.csv`.

Install packages in your selected Python environment if needed:

    python -m pip install numpy pandas matplotlib torch

Inputs: six hourly cloud masks, six hourly cloud-top-height maps, and the
corresponding six hours of station weather. Targets: the next-hour cloud
mask and cloud-top-height map. Rainfall is NOT used. The CSV readings at the
target hour are never passed into the model.

This is a new experiment. Its checkpoint and plot are saved under
`outputs/cloud_map_with_stations/`, leaving earlier models untouched.


### Imports and settings


In [10]:
from __future__ import annotations

import argparse
from pathlib import Path

import matplotlib
matplotlib.use("Agg")  # Works from a terminal without a display.
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

try:
    import cartopy.crs as ccrs
except ImportError:
    ccrs = None  # Still produces a map when Cartopy is not installed.


STATION_FILES = (
    "hly3723_subset.csv",  # Casement
    "hly3904_subset.csv",  # Cork Airport
    "hly4935_subset.csv",  # Knock Airport
    "hly518_subset.csv",   # Shannon Airport
    "hly532.csv",          # Dublin Airport
)
CSV_COLUMNS = ("date", "wdsp", "wddir", "rhum", "msl", "temp", "clamt")
WEATHER_COLUMNS = ("wdsp", "rhum", "msl", "temp", "clamt", "wind_sin", "wind_cos")


def find_data_file(data_dir: Path, filename: str) -> Path:
    matches = sorted(data_dir.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} below {data_dir.resolve()}")
    return matches[0]


def read_station(path: Path, times: np.ndarray) -> pd.DataFrame:
    # Met Éireann hourly CSVs start with a description, then a 'date,...' header.
    with path.open(encoding="utf-8-sig", errors="replace") as file:
        header_line = next(
            (line_number for line_number, line in enumerate(file)
             if line.startswith("date,")),
            None,
        )
    if header_line is None:
        raise ValueError(f"No date header found in {path}")

    station = pd.read_csv(
        path, skiprows=header_line, usecols=list(CSV_COLUMNS), low_memory=False
    )
    station["date"] = pd.to_datetime(
        station["date"], format="%d-%b-%Y %H:%M", errors="coerce"
    )
    station = station.dropna(subset=["date"]).drop_duplicates("date")
    station = station.set_index("date").reindex(pd.DatetimeIndex(times))

    # Keep valid values such as 999 hPa. Discard impossible readings and
    # placeholder values in other fields using the physically valid ranges.
    ranges = {
        "wdsp": (0, 120), "wddir": (0, 360), "rhum": (0, 100),
        "msl": (850, 1100), "temp": (-35, 45), "clamt": (0, 8),
    }
    for name, (low, high) in ranges.items():
        values = pd.to_numeric(station[name], errors="coerce")
        station[name] = values.where(values.between(low, high))

    # Wind direction is circular: 360 degrees and 0 degrees are equivalent.
    radians = np.deg2rad(station["wddir"])
    station["wind_sin"] = np.sin(radians)
    station["wind_cos"] = np.cos(radians)
    return station[list(WEATHER_COLUMNS)]


def load_data(data_dir: Path):
    if data_dir.is_file():
        # Accept a direct path to the cloud .npz file too.
        npz_file = data_dir
        # For data/raw/satellite/<file>, search the project for station CSVs.
        if data_dir.parent.name == "satellite" and data_dir.parent.parent.name == "raw":
            station_search_dir = data_dir.parents[3]
        else:
            station_search_dir = data_dir.parent
    else:
        npz_file = find_data_file(
            data_dir, "ireland_clouds_20240101_20250101.npz"
        )
        station_search_dir = data_dir
    with np.load(npz_file) as data:
        times = data["times"]
        masks = data["cloud_mask"]
        heights = data["cth_km"]
        lat = data["lat"]
        lon = data["lon"]

    if masks.shape != heights.shape or masks.shape[0] != len(times):
        raise ValueError("Cloud masks, heights, and timestamps do not align")
    print(f"Cloud data: {npz_file} | {masks.shape}", flush=True)

    stations = []
    for filename in STATION_FILES:
        path = find_data_file(station_search_dir, filename)
        station = read_station(path, times)
        stations.append(station.to_numpy(dtype=np.float32))
        print(f"Station: {filename} | {station.notna().all(axis=1).sum()} complete hours")

    weather = np.stack(stations, axis=1)  # hours, five stations, seven fields
    return times, masks, heights, lat, lon, weather


### Build exactly the same time splits as the earlier cloud experiment


In [11]:
def make_splits(times: np.ndarray, masks: np.ndarray):
    valid_pixels = np.count_nonzero(masks != 3, axis=(1, 2))
    starts = np.arange(len(times) - 6)
    sequences = starts[:, None] + np.arange(6)
    targets = starts + 6

    usable = (
        (valid_pixels[sequences] > 0).all(axis=1)
        & (valid_pixels[targets] > 0)
    )
    sequences = sequences[usable]
    targets = targets[usable]

    train_end = int(0.70 * len(times))
    validation_end = int(0.85 * len(times))
    train_rows = np.flatnonzero(targets < train_end)
    validation_rows = np.flatnonzero(
        (targets >= train_end + 6) & (targets < validation_end)
    )
    test_rows = np.flatnonzero(targets >= validation_end + 6)

    for name, rows in (("Train", train_rows), ("Validation", validation_rows),
                       ("Test", test_rows)):
        print(f"{name}: {len(rows)} | {times[targets[rows[0]]]} to "
              f"{times[targets[rows[-1]]]}")
    return sequences, targets, train_rows, validation_rows, test_rows, train_end


def prepare_station_weather(weather: np.ndarray, train_end: int) -> np.ndarray:
    # Compute all imputation/normalization statistics using training hours only.
    training = weather[:train_end]
    mean = np.nanmean(training, axis=(0, 1))
    std = np.nanstd(training, axis=(0, 1))
    if not np.isfinite(mean).all():
        raise ValueError("A station feature has no valid readings in training")
    std = np.where(std < 1e-6, 1.0, std)
    missing = ~np.isfinite(weather)
    weather = np.where(missing, mean[None, None, :], weather)
    weather = (weather - mean[None, None, :]) / std[None, None, :]
    # The last channel tells the model if the original pressure was missing.
    pressure_missing = missing[:, :, 2:3].astype(np.float32)
    return np.concatenate([weather, pressure_missing], axis=2).astype(np.float32)


### Dataset: six maps and station readings -> next-hour maps


In [12]:
class CloudStationDataset(Dataset):
    def __init__(self, rows, sequences, targets, masks, heights, weather):
        self.rows = rows
        self.sequences = sequences
        self.targets = targets
        self.masks = masks
        self.heights = heights
        self.weather = weather

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        hours = self.sequences[row]
        target_hour = self.targets[row]

        input_masks = self.masks[hours]
        cloud = (input_masks == 2).astype(np.float32)
        valid = (input_masks != 3).astype(np.float32)
        height = self.heights[hours].astype(np.float32)
        height = np.nan_to_num(height, nan=0.0, posinf=0.0, neginf=0.0)
        height = np.clip(height, 0, 12) / 12 * cloud
        images = np.stack((cloud, height, valid), axis=1)

        target_mask = self.masks[target_hour]
        raw_height = self.heights[target_hour].astype(np.float32)
        target_cloud = (target_mask == 2).astype(np.float32)[None]
        target_valid = (target_mask != 3).astype(np.float32)[None]
        height_valid = ((target_mask == 2) & np.isfinite(raw_height)).astype(np.float32)[None]
        target_height = np.clip(
            np.nan_to_num(raw_height, nan=0.0, posinf=0.0, neginf=0.0),
            0, 12,
        )[None] / 12
        target_height *= target_cloud

        return (
            torch.from_numpy(images),
            torch.from_numpy(self.weather[hours]),  # no target-hour CSV data
            torch.from_numpy(target_cloud),
            torch.from_numpy(target_height),
            torch.from_numpy(target_valid),
            torch.from_numpy(height_valid),
        )


### CNN–LSTM with full-resolution input detail and station-weather branch


In [13]:
class CloudStationModel(nn.Module):
    def __init__(self, station_count=5, weather_fields=8):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 8, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(8, 16, 3, stride=2, padding=1), nn.ReLU(),
        )
        self.image_lstm = nn.LSTM(16, 16, batch_first=True)
        self.station_lstm = nn.LSTM(
            station_count * weather_fields, 16, batch_first=True
        )
        # The last image reaches the decoder at its original 90 x 120 size.
        self.decoder = nn.Sequential(
            nn.Conv2d(32 + 3, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(),
        )
        self.cloud_head = nn.Conv2d(16, 1, 1)
        self.height_head = nn.Conv2d(16, 1, 1)

    def forward(self, images, weather):
        batch, hours, channels, height, width = images.shape
        encoded = self.cnn(images.reshape(batch * hours, channels, height, width))
        _, features, small_h, small_w = encoded.shape
        encoded = encoded.reshape(batch, hours, features, small_h, small_w)
        sequence = encoded.permute(0, 3, 4, 1, 2).reshape(
            batch * small_h * small_w, hours, features
        )
        sequence, _ = self.image_lstm(sequence)
        spatial = sequence[:, -1].reshape(batch, small_h, small_w, 16)
        spatial = spatial.permute(0, 3, 1, 2)

        station_sequence = weather.reshape(batch, hours, -1)
        station_sequence, _ = self.station_lstm(station_sequence)
        station_features = station_sequence[:, -1, :, None, None]
        station_features = station_features.expand(-1, -1, small_h, small_w)

        combined = torch.cat((spatial, station_features), dim=1)
        combined = F.interpolate(
            combined, size=(height, width), mode="bilinear", align_corners=False
        )
        last_image = images[:, -1]
        decoded = self.decoder(torch.cat((combined, last_image), dim=1))

        cloud_logits = 4 * last_image[:, 0:1] - 2 + self.cloud_head(decoded)

        # Start the height forecast from the last recorded height where there
        # was cloud; start newly cloudy pixels from a neutral 6 km estimate.
        prior = torch.where(
            last_image[:, 0:1] > 0.5,
            last_image[:, 1:2],
            torch.full_like(last_image[:, 1:2], 0.5),
        ).clamp(0.01, 0.99)
        height_prediction = torch.sigmoid(torch.logit(prior) + self.height_head(decoded))
        return cloud_logits, height_prediction


def loss_function(logits, predicted_height, cloud, height, valid, height_valid):
    cloud_loss = F.binary_cross_entropy_with_logits(
        logits[valid.bool()], cloud[valid.bool()]
    )
    cloudy_pixels = height_valid.bool()
    if cloudy_pixels.any():
        height_loss = F.smooth_l1_loss(
            predicted_height[cloudy_pixels], height[cloudy_pixels]
        )
    else:
        height_loss = cloud_loss.new_zeros(())
    return cloud_loss + 5.0 * height_loss


@torch.no_grad()
def evaluate(model, loader, device, baseline=False):
    model.eval()
    total_loss = intersection = union = height_error = height_count = 0.0
    for batch in loader:
        images, weather, cloud, height, valid, height_valid = (
            item.to(device) for item in batch
        )
        if baseline:
            predicted_cloud = images[:, -1, 0:1] > 0.5
            predicted_height = images[:, -1, 1:2]
        else:
            logits, predicted_height = model(images, weather)
            total_loss += loss_function(
                logits, predicted_height, cloud, height, valid, height_valid
            ).item() * images.shape[0]
            predicted_cloud = torch.sigmoid(logits) >= 0.5

        good = valid.bool()
        intersection += ((predicted_cloud & cloud.bool()) & good).sum().item()
        union += ((predicted_cloud | cloud.bool()) & good).sum().item()
        height_good = height_valid.bool()
        height_error += torch.abs(
            predicted_height[height_good] - height[height_good]
        ).sum().item() * 12
        height_count += height_good.sum().item()

    return {
        "loss": total_loss / len(loader.dataset) if not baseline else float("nan"),
        "iou": 100 * intersection / union,
        "height_mae": height_error / height_count,
    }


### Plot using the established sea / land / purple-height palette


In [14]:
@torch.no_grad()
def save_example(model, dataset, example_number, times, masks, heights, lat, lon,
                 device, output_file):
    if example_number >= len(dataset):
        raise ValueError(f"Example {example_number} exceeds test set size")
    row = dataset.rows[example_number]
    hours = dataset.sequences[row]
    next_hour = dataset.targets[row]
    images, weather, *_ = dataset[example_number]
    scores, predicted_h = model(
        images[None].to(device), weather[None].to(device)
    )
    predicted_cloud = (torch.sigmoid(scores[0, 0]) >= 0.5).cpu().numpy()
    predicted_h = (predicted_h[0, 0] * 12).cpu().numpy()

    # The land/sea background is inferred only from training-period maps.
    training_end = int(0.70 * len(times))
    land = np.any(masks[:training_end] == 1, axis=0)
    predicted_mask = np.where(land, 1, 0).astype(np.uint8)
    predicted_mask[predicted_cloud] = 2
    predicted_mask[masks[hours[-1]] == 3] = 3

    background = ListedColormap(("#0161c2", "#09773e", "#8e63b9", "#ffffff"))
    category_norm = BoundaryNorm((-0.5, 0.5, 1.5, 2.5, 3.5), background.N)
    map_projection = {"projection": ccrs.PlateCarree()} if ccrs else {}
    fig, axes = plt.subplots(2, 4, figsize=(14, 8), subplot_kw=map_projection)
    fig.subplots_adjust(left=0.04, right=0.86, bottom=0.08, top=0.90,
                        wspace=0.19, hspace=0.22)

    for panel, ax in enumerate(axes.flat):
        if panel < 6:
            time_position = hours[panel]
            panel_mask = masks[time_position]
            panel_height = heights[time_position].astype(np.float32)
            title = f"Input {panel + 1}"
        elif panel == 6:
            time_position = next_hour
            panel_mask = masks[time_position]
            panel_height = heights[time_position].astype(np.float32)
            title = "Recorded next hour"
        else:
            time_position = next_hour
            panel_mask = predicted_mask
            panel_height = predicted_h
            title = "CNN–LSTM predicted next hour"

        transform = {"transform": ccrs.PlateCarree()} if ccrs else {}
        ax.pcolormesh(lon, lat, panel_mask, cmap=background, norm=category_norm,
                      shading="auto", rasterized=True, **transform)
        cloudy_height = np.ma.masked_where(panel_mask != 2, panel_height)
        coloured_clouds = ax.pcolormesh(
            lon, lat, cloudy_height, cmap="Purples", vmin=0, vmax=12,
            shading="auto", rasterized=True, **transform
        )
        if ccrs:
            ax.coastlines(resolution="50m", color="black", linewidth=0.7)
        ax.set_xlim(lon.min(), lon.max())
        ax.set_ylim(lat.min(), lat.max())
        ax.set_aspect("auto")
        ax.set_title(f"{title}: {str(times[time_position])[:16]} UTC", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])

    fig.suptitle("Six cloud maps → recorded and predicted next-hour maps", fontsize=15)
    fig.legend(
        handles=(Patch(color="#0161c2", label="Clear sea"),
                 Patch(color="#09773e", label="Clear land"),
                 Patch(facecolor="#ffffff", edgecolor="black", label="No data")),
        loc="lower center", ncol=3, frameon=False,
    )
    fig.colorbar(
        coloured_clouds, cax=fig.add_axes((0.89, 0.17, 0.025, 0.67)),
        label="Cloud-top height (km)",
    )
    fig.savefig(output_file, dpi=160, bbox_inches="tight")
    plt.close(fig)
    print(f"Comparison image: {output_file.resolve()}")


### Run the experiment


Set `DATA_DIR` to your project root **or** the path to the cloud `.npz` file. The notebook searches the project for the five station CSV files. Then run the cell.


In [15]:
DATA_DIR =  Path(
    "data/raw/satellite/ireland_clouds_20240101_20250101.npz"
)  # Project folder, or direct path to the cloud .npz file.

def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data-dir", type=Path, default=DATA_DIR)
    parser.add_argument("--epochs", type=int, default=8)
    parser.add_argument("--batch-size", type=int, default=8)
    parser.add_argument("--example", type=int, default=100)
    parser.add_argument("--output-dir", type=Path,
                        default=Path("outputs/cloud_map_with_stations"))
    args = parser.parse_args([])
    args.output_dir.mkdir(parents=True, exist_ok=True)

    torch.manual_seed(42)
    np.random.seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}", flush=True)

    times, masks, heights, lat, lon, raw_weather = load_data(args.data_dir)
    sequences, targets, train_rows, val_rows, test_rows, train_end = make_splits(
        times, masks
    )
    weather = prepare_station_weather(raw_weather, train_end)
    print(f"Station input shape: {weather.shape} (hours, stations, features)")

    def loader(rows, shuffle=False):
        dataset = CloudStationDataset(rows, sequences, targets, masks, heights, weather)
        return DataLoader(dataset, batch_size=args.batch_size, shuffle=shuffle,
                          num_workers=0)

    train_loader = loader(train_rows, shuffle=True)
    val_loader = loader(val_rows)
    test_loader = loader(test_rows)
    model = CloudStationModel(weather_fields=weather.shape[-1]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    checkpoint = args.output_dir / "best_cloud_station_model.pt"

    best_val_loss = float("inf")
    stale_epochs = 0
    for epoch in range(1, args.epochs + 1):
        model.train()
        train_loss = 0.0
        for batch in train_loader:
            images, station_weather, cloud, height, valid, height_valid = (
                item.to(device) for item in batch
            )
            optimizer.zero_grad()
            scores, predicted_height = model(images, station_weather)
            loss = loss_function(scores, predicted_height, cloud, height,
                                 valid, height_valid)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * len(images)

        val = evaluate(model, val_loader, device)
        print(
            f"Epoch {epoch:02d} | train loss {train_loss / len(train_rows):.4f} | "
            f"validation loss {val['loss']:.4f} | cloud IoU {val['iou']:.1f}% | "
            f"height MAE {val['height_mae']:.2f} km", flush=True
        )
        if val["loss"] < best_val_loss:
            best_val_loss = val["loss"]
            stale_epochs = 0
            torch.save(model.state_dict(), checkpoint)
            print("  Saved best model", flush=True)
        else:
            stale_epochs += 1
            if stale_epochs >= 3:
                print("Stopped: validation loss has not improved for three epochs")
                break

    model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))
    model.eval()
    test = evaluate(model, test_loader, device)
    last_map = evaluate(model, test_loader, device, baseline=True)
    print(f"CNN–LSTM test | cloud IoU {test['iou']:.1f}% | "
          f"height MAE {test['height_mae']:.2f} km")
    print(f"Last-map baseline | cloud IoU {last_map['iou']:.1f}% | "
          f"height MAE {last_map['height_mae']:.2f} km")
    print("Previous cloud-only experiment: test cloud IoU 84.5%")
    print("Compare scores on the same dates, then inspect the saved image.")

    save_example(model, test_loader.dataset, args.example, times, masks,
                 heights, lat, lon, device, args.output_dir / "comparison.png")
    print(f"Model checkpoint: {checkpoint.resolve()}")


main()


Device: cpu
Cloud data: data\raw\satellite\ireland_clouds_20240101_20250101.npz | (8784, 90, 120)
Station: hly3723_subset.csv | 8784 complete hours
Station: hly3904_subset.csv | 8784 complete hours
Station: hly4935_subset.csv | 8784 complete hours
Station: hly518_subset.csv | 8784 complete hours
Station: hly532.csv | 8784 complete hours
Train: 6104 | 2024-01-01T06:00:00.000000 to 2024-09-13T03:00:00.000000
Validation: 1312 | 2024-09-13T10:00:00.000000 to 2024-11-07T01:00:00.000000
Test: 1312 | 2024-11-07T08:00:00.000000 to 2024-12-31T23:00:00.000000
Station input shape: (8784, 5, 8) (hours, stations, features)
Epoch 01 | train loss 0.4155 | validation loss 0.4184 | cloud IoU 83.7% | height MAE 1.56 km
  Saved best model
Epoch 02 | train loss 0.3698 | validation loss 0.3798 | cloud IoU 84.3% | height MAE 1.38 km
  Saved best model
Epoch 03 | train loss 0.3588 | validation loss 0.3730 | cloud IoU 84.4% | height MAE 1.40 km
  Saved best model
Epoch 04 | train loss 0.3541 | validation loss